# Notebook 007. Spatial autocorrelation summary
-------

Visualise the spatial-autocorrelation analysis. The analysis is done by [`scripts/run_autocorrelation.py`](../scripts/run_autocorrelation.py), which writes four tidy tables to `results/autocorrelation/`; this notebook only summarises them.

The first code cell also adds residual autocorrelation of each completed nested-CV model run.

In [ ]:
# Setup. Resolve paths, import the shared autocorrelation primitives and the
# project conventions, and pin the output schemas. The schemas mirror
# scripts/run_autocorrelation.py so appended residual rows validate against
# the same contract as the tables already on disk.
import json
import logging
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed

from utils.autocorrelation import (
    MORAN_THRESHOLDS_M,
    N_SAMPLE,
    VGRAM_MODELS,
    VGRAM_USE_NUGGET,
    fit_variograms,
    moran_correlogram,
)
from utils.io import ColumnSpec, Schema, write_csv
from utils.paths import get_project_paths
from utils.terminology import FOLD_IDS, SEED

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True
)
logger = logging.getLogger("007_autocorrelation")

NOTEBOOK = "007_autocorrelation"
RESIDUAL_GROUP = "residual"

paths = get_project_paths()
AUTOCORR_DIR = paths.results / "autocorrelation"
NESTED_CV_DIR = paths.results / "main_nested_cv"

# The four tidy tables written by the Stage 5 script. Residual rows are appended
# to these in place by the update cell, then summarised by the table cell.
FITS_CSV = AUTOCORR_DIR / "variogram_fits.csv"
EMPIRICAL_CSV = AUTOCORR_DIR / "variogram_empirical.csv"
FITTED_CSV = AUTOCORR_DIR / "variogram_fitted.csv"
MORAN_CSV = AUTOCORR_DIR / "moran_correlogram.csv"

_FLOAT, _STR, _INT, _BOOL = (
    ColumnSpec("float64"),
    ColumnSpec("object"),
    ColumnSpec("int64"),
    ColumnSpec("bool"),
)
VARIOGRAM_FITS_SCHEMA: Schema = {
    "variable": _STR,
    "group": _STR,
    "is_pc": _BOOL,
    "model": _STR,
    "effective_range_m": _FLOAT,
    "sill": _FLOAT,
    "nugget": _FLOAT,
    "nugget_sill_ratio": _FLOAT,
    "rmse": _FLOAT,
    "range_hits_maxlag": _BOOL,
    "n": _INT,
    "maxlag_m": _FLOAT,
    "n_lags": _INT,
}
VARIOGRAM_EMPIRICAL_SCHEMA: Schema = {
    "variable": _STR,
    "group": _STR,
    "lag_m": _FLOAT,
    "semivariance": _FLOAT,
}
VARIOGRAM_FITTED_SCHEMA: Schema = {
    "variable": _STR,
    "group": _STR,
    "model": _STR,
    "lag_m": _FLOAT,
    "semivariance": _FLOAT,
}
MORAN_SCHEMA: Schema = {
    "variable": _STR,
    "group": _STR,
    "threshold_m": _INT,
    "morans_i": _FLOAT,
    "expected_i": _FLOAT,
    "z_sim": _FLOAT,
    "p_sim": _FLOAT,
    "p_norm": _FLOAT,
    "n": _INT,
    "mean_neighbours": _FLOAT,
}

print(f"[setup] autocorrelation tables: {AUTOCORR_DIR.relative_to(paths.repo_root)}")
print(f"[setup] nested-CV runs:         {NESTED_CV_DIR.relative_to(paths.repo_root)}")
print(
    f"[setup] nugget fitted (not forced to 0): VGRAM_USE_NUGGET={VGRAM_USE_NUGGET}; "
    f"pixel residuals sampled to N_SAMPLE={N_SAMPLE} (seed={SEED})"
)

## Residual spatial autocorrelation for completed model runs

Each completed nested cross-validation run under `results/main_nested_cv/` contributes:

- **`{model}__{feature_set}__parcel`:** one residual per labelled parcel, `y_true - p_mean`, at the parcel centroid (EPSG:3035). Same support as the `ogf_label` variogram (all labelled parcels), so its range and Moran's I are directly comparable to the label's.
- **`{model}__{feature_set}__pixel`:** the pooled out-of-fold pixel residual `y_true - p`, uniformly sampled to `N_SAMPLE` points (seed `SEED`). Same sampling as the predictor bands, so it is comparable to them; it captures within-parcel/short-range structure the parcel centroids average away.

In [ ]:
# Update. Discover completed runs, build each one's parcel- and pixel-support
# residual variables, analyse them with the shared primitives, and append the
# fragments to the four tables. Safe to re-run: by default only runs missing a
# support are computed; existing rows are preserved (and old-scheme rows cleaned).
FORCE_RECOMPUTE = False
SUPPORTS = ("parcel", "pixel")


def discover_completed_runs(nested_cv_dir: Path) -> dict[str, tuple[str, Path, list[str]]]:
    """Latest fully-complete run per (model, feature_set): all folds done, predictions present."""
    runs: dict[str, tuple[str, Path, list[str]]] = {}
    for run_dir in sorted(nested_cv_dir.glob("*__*__*")):
        status_path = run_dir / "fold_status.json"
        if not status_path.exists():
            continue
        status = json.loads(status_path.read_text())
        # A run in progress lists only the folds finished so far, so require every fold.
        if set(status) != {str(fold) for fold in FOLD_IDS} or not all(
            fold.get("complete") for fold in status.values()
        ):
            continue
        fold_keys = list(status.keys())
        have_parcel = all(
            (run_dir / f"parcel_predictions_fold{k}.parquet").exists() for k in fold_keys
        )
        have_pixel = all((run_dir / f"predictions_fold{k}.parquet").exists() for k in fold_keys)
        if not (have_parcel and have_pixel):
            continue
        timestamp, model, feature_set = run_dir.name.split("__", 2)
        key = f"{model}__{feature_set}"
        if key not in runs or runs[key][0] < timestamp:
            runs[key] = (timestamp, run_dir, fold_keys)
    return runs


def residual_run(variable: str) -> str | None:
    """The run a residual variable belongs to, or None if it is not '<run>__<support>'."""
    run, sep, support = variable.rpartition("__")
    return run if (sep and support in SUPPORTS) else None


def parcel_residual_points(run_dir: Path, fold_keys: list[str], centroids: pd.DataFrame):
    """Pooled out-of-fold parcel residual (y_true - p_mean) at parcel centroids."""
    pooled = pd.concat(
        [pd.read_parquet(run_dir / f"parcel_predictions_fold{k}.parquet") for k in fold_keys],
        ignore_index=True,
    ).drop_duplicates("parcel_id")
    joined = pooled.join(centroids, on="parcel_id").dropna(subset=["x", "y"])
    coords = joined[["x", "y"]].to_numpy(dtype=float)
    values = (joined["y_true"] - joined["p_mean"]).to_numpy(dtype=float)
    return coords, values


def pixel_residual_points(
    run_dir: Path, fold_keys: list[str], pixel_xy: pd.DataFrame, n_sample: int, seed: int
):
    """Pooled out-of-fold pixel residual (y_true - p), uniformly sampled to n_sample points."""
    pooled = pd.concat(
        [
            pd.read_parquet(
                run_dir / f"predictions_fold{k}.parquet", columns=["pixel_id", "y_true", "p"]
            )
            for k in fold_keys
        ],
        ignore_index=True,
    ).drop_duplicates("pixel_id")
    if len(pooled) > n_sample:
        rng = np.random.default_rng(seed)
        pooled = pooled.iloc[rng.choice(len(pooled), size=n_sample, replace=False)]
    sample = pooled.merge(pixel_xy, on="pixel_id", how="inner")
    coords = sample[["x", "y"]].to_numpy(dtype=float)
    values = (sample["y_true"] - sample["p"]).to_numpy(dtype=float)
    return coords, values


def analyse_residual(name: str, coords, values):
    """Variogram + Moran fragments for one residual variable, in the four-table schema."""
    result = fit_variograms(coords, values)
    fits = result.fits.copy()
    fits.insert(0, "variable", name)
    fits.insert(1, "group", RESIDUAL_GROUP)
    fits.insert(2, "is_pc", False)
    fits["n"] = result.n
    fits["maxlag_m"] = result.maxlag_m
    fits["n_lags"] = result.n_lags
    empirical = pd.DataFrame(
        {
            "variable": name,
            "group": RESIDUAL_GROUP,
            "lag_m": result.lags_m,
            "semivariance": result.semivariance,
        }
    )
    fitted = pd.concat(
        [
            pd.DataFrame(
                {
                    "variable": name,
                    "group": RESIDUAL_GROUP,
                    "model": model,
                    "lag_m": result.curve_lag_m,
                    "semivariance": curve,
                }
            )
            for model, curve in result.fitted.items()
        ],
        ignore_index=True,
    )
    moran = moran_correlogram(coords, values).reset_index()
    moran.insert(0, "variable", name)
    moran.insert(1, "group", RESIDUAL_GROUP)
    return fits, empirical, fitted, moran


def _coerce(frame: pd.DataFrame, schema: Schema) -> pd.DataFrame:
    out = frame.copy()
    for column, spec in schema.items():
        dtype = spec.dtype if isinstance(spec.dtype, str) else spec.dtype[0]
        out[column] = out[column].astype(dtype)
    return out


def append_fragments(
    csv_path: Path, schema: Schema, new_frame: pd.DataFrame, replace_runs: set[str]
):
    """Append residual rows: drop rows of the recomputed runs (and any old-scheme residual
    rows), keep everything else, then write validated against the schema."""
    existing = pd.read_csv(csv_path)
    is_residual = existing["group"] == RESIDUAL_GROUP
    runs_of = existing["variable"].map(residual_run)
    stale = is_residual & (runs_of.isin(replace_runs) | runs_of.isna())
    combined = pd.concat([existing[~stale], new_frame[list(schema)]], ignore_index=True)
    write_csv(_coerce(combined, schema), csv_path, schema)


runs = discover_completed_runs(NESTED_CV_DIR) if NESTED_CV_DIR.exists() else {}
existing_residuals = set()
if FITS_CSV.exists():
    fits_now = pd.read_csv(FITS_CSV)
    existing_residuals = set(fits_now.loc[fits_now["group"] == RESIDUAL_GROUP, "variable"])


def run_done(run: str) -> bool:
    return all(f"{run}__{support}" in existing_residuals for support in SUPPORTS)


todo = {run: info for run, info in runs.items() if FORCE_RECOMPUTE or not run_done(run)}

print(
    f"[update] {len(runs)} completed run(s); "
    f"{sum(run_done(r) for r in runs)} already analysed (both supports); "
    f"{len(todo)} to compute (FORCE_RECOMPUTE={FORCE_RECOMPUTE})."
)

if todo:
    pixels = pd.read_parquet(
        NESTED_CV_DIR / "pixel_index.parquet", columns=["pixel_id", "parcel_id", "x", "y"]
    )
    centroids = pixels.groupby("parcel_id")[["x", "y"]].mean()
    pixel_xy = pixels[["pixel_id", "x", "y"]]

    # I/O and sampling in the main process (cheap); fan out only the heavy
    # variogram + Moran work, which needs just the small (coords, values) arrays.
    tasks: list[tuple[str, np.ndarray, np.ndarray]] = []
    for run, (_timestamp, run_dir, fold_keys) in todo.items():
        parcel_coords, parcel_values = parcel_residual_points(run_dir, fold_keys, centroids)
        tasks.append((f"{run}__parcel", parcel_coords, parcel_values))
        pixel_coords, pixel_values = pixel_residual_points(
            run_dir, fold_keys, pixel_xy, N_SAMPLE, SEED
        )
        tasks.append((f"{run}__pixel", pixel_coords, pixel_values))
        logger.info(
            "Residual %s: %d parcels, %d sampled pixels.",
            run,
            len(parcel_values),
            len(pixel_values),
        )

    fragments = Parallel(n_jobs=min(len(tasks), os.cpu_count() or 1))(
        delayed(analyse_residual)(name, coords, values) for name, coords, values in tasks
    )
    replace_runs = set(todo)
    append_fragments(
        FITS_CSV,
        VARIOGRAM_FITS_SCHEMA,
        pd.concat([f[0] for f in fragments], ignore_index=True),
        replace_runs,
    )
    append_fragments(
        EMPIRICAL_CSV,
        VARIOGRAM_EMPIRICAL_SCHEMA,
        pd.concat([f[1] for f in fragments], ignore_index=True),
        replace_runs,
    )
    append_fragments(
        FITTED_CSV,
        VARIOGRAM_FITTED_SCHEMA,
        pd.concat([f[2] for f in fragments], ignore_index=True),
        replace_runs,
    )
    append_fragments(
        MORAN_CSV,
        MORAN_SCHEMA,
        pd.concat([f[3] for f in fragments], ignore_index=True),
        replace_runs,
    )
    print(f"[update] appended parcel+pixel residual rows for: {', '.join(sorted(replace_runs))}")
else:
    print("[update] nothing to do; residual tables are up to date.")

## Comprehensive autocorrelation summary table

One row per variable — every individual band, embedding dimension, embedding principal component, the old-growth label, and a parcel- and a pixel-support residual per completed model run. Columns are grouped into blocks:

- **info** — `group`, `is_pc`, `n` (points the variogram and Moran were built from) and the best-fitting model (lowest RMSE) with its effective range in km.
- **spherical / exponential / gaussian** — each theoretical variogram model's effective `range_km`, `sill`, `nugget`, `nugget_sill_ratio` and fit `rmse`. The nugget is a **free fitted parameter** (`VGRAM_USE_NUGGET = True`): the curves are *not* forced through nugget = 0, so a non-zero `nugget` (and `nugget_sill_ratio`) reflects fitted micro-scale variance plus noise.
- **moran** — at each distance band (5 / 10 / 15 / 20 km):
  - **`I`** — Moran's I (`morans_i`): the correlation between each unit's value and the mean of its neighbours within the band. ≈ 0 is no spatial autocorrelation (the null expectation is `E[I] = −1/(n−1)`, just below 0); **positive means nearby units are alike** (clustering). It decays toward 0 as the band widens.
  - **`z`** — the permutation z-score (`z_sim`): how many standard deviations the observed `I` lies above the mean of 999 spatially-random reshuffles. A large `z` means the clustering is far stronger than chance.
  - **`p`** — the permutation pseudo p-value (`p_sim`): the fraction of permutations whose `I` matched or exceeded the observed value, floored at `1/(999+1) = 0.001`. `p = 0.001` means no random permutation reached the observed `I`.

The full long tables — including `expected_i`, the normal-theory `p_norm`, `mean_neighbours`, and the empirical and fitted variogram curves used for plotting are in `results/autocorrelation/`. The wide table here is also written to `figures/007_autocorrelation/autocorrelation_summary_table.csv`.

In [ ]:
# Comprehensive table. Pivot the per-model variogram fits and the per-threshold
# Moran correlogram into one wide, grouped table, ordered label -> predictors ->
# residuals. Displayed in full and saved as a single flat CSV artifact.
from utils.autocorrelation import range_unresolved

fits = pd.read_csv(FITS_CSV)
moran = pd.read_csv(MORAN_CSV)
fits["range_km"] = fits["effective_range_m"] / 1_000.0

# A fitted range is "not resolved" when it runs into either boundary of the
# empirical window (see utils.autocorrelation.range_unresolved): the ceiling
# (>= 90% of the max lag, ~36 km), where the optimiser parks the range near the
# maximum lag, or the floor (below the first lag class, maxlag / n_lags ~ 1 km),
# where the fit collapses inside the first bin to a pure nugget. Such rows get no
# numeric best range or best model. n_lags is passed explicitly so the floor
# stays correct if the binning changes.
fits["range_unresolved"] = range_unresolved(
    fits["effective_range_m"], fits["maxlag_m"], n_lags=fits["n_lags"]
)

MODELS = list(VGRAM_MODELS)
THRESHOLDS_KM = [threshold // 1_000 for threshold in MORAN_THRESHOLDS_M]
VGRAM_VALUES = ["range_km", "sill", "nugget", "nugget_sill_ratio", "rmse"]
MORAN_SHORT = {"morans_i": "I", "z_sim": "z", "p_sim": "p"}

meta = fits.drop_duplicates("variable").set_index("variable")[["group", "is_pc", "n"]]
n_resolved = (~fits["range_unresolved"]).groupby(fits["variable"]).sum().rename("n_models_resolved")
# Best model / best range from the resolved fits only; variables with no resolved
# fit are absent here and so join to NaN (best_model / best_range_km suppressed).
resolved = fits[~fits["range_unresolved"]]
best = (
    resolved.loc[resolved.groupby("variable")["rmse"].idxmin()]
    .set_index("variable")[["model", "range_km"]]
    .rename(columns={"model": "best_model", "range_km": "best_range_km"})
)

vgram = fits.pivot_table(index="variable", columns="model", values=VGRAM_VALUES)
vgram.columns = [f"{model}.{value}" for value, model in vgram.columns]
vgram = vgram[[f"{model}.{value}" for model in MODELS for value in VGRAM_VALUES]]

mwide = moran.pivot_table(index="variable", columns="threshold_m", values=list(MORAN_SHORT))
mwide.columns = [
    f"moran.{MORAN_SHORT[stat]}@{threshold // 1000}km" for stat, threshold in mwide.columns
]
mwide = mwide[[f"moran.{stat}@{km}km" for km in THRESHOLDS_KM for stat in ("I", "z", "p")]]

GROUP_ORDER = ["label", "baseline", "conventional_eo", "alphaearth", "tessera", RESIDUAL_GROUP]
summary = meta.join(n_resolved).join(best).join(vgram).join(mwide)
summary = (
    summary.assign(
        _order=summary["group"].map({g: i for i, g in enumerate(GROUP_ORDER)}).fillna(99)
    )
    .sort_values(["_order", "group", "variable"])
    .drop(columns="_order")
)

figures_dir = paths.figures / NOTEBOOK
figures_dir.mkdir(parents=True, exist_ok=True)
summary_path = figures_dir / "autocorrelation_summary_table.csv"
summary.to_csv(summary_path)
print(
    f"[table] {len(summary)} variables x {summary.shape[1]} columns "
    f"({(summary['group'] == RESIDUAL_GROUP).sum()} residual; "
    f"{int((summary['n_models_resolved'] == 0).sum())} with range not resolved) -> "
    f"{summary_path.relative_to(paths.repo_root)}"
)


def _group_columns(columns):
    """Lift the flat 'block.field' names into a two-level (block, field) MultiIndex."""
    tuples = []
    for column in columns:
        head = column.split(".", 1)[0]
        if head in MODELS or head == "moran":
            tuples.append((head, column.split(".", 1)[1]))
        else:
            tuples.append(("info", column))
    return pd.MultiIndex.from_tuples(tuples)


display_table = summary.copy()
display_table.columns = _group_columns(summary.columns)

In [ ]:
# View. Residual rows first (the dynamically-populated part), then every variable.
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 240)

residual_view = display_table[display_table[("info", "group")] == RESIDUAL_GROUP]
print(
    f"Residual autocorrelation for {len(residual_view)} variable(s) "
    f"({residual_view.shape[0] // 2} run(s) x parcel/pixel):"
)
display(residual_view.round(3))

print("Full autocorrelation summary (all variables):")
display(display_table.round(3))

## Range snapshot (presentation)

In [ ]:
# Range snapshot (presentation). Dumbbell of each variable's effective variogram range
# (km) across the three theoretical models, for the old-growth label, an illustrative
# sample of the named predictors, and the XGBoost + TESSERA out-of-fold parcel residual.
# Embedding dimensions and PCs are omitted (their dimensions have no plain-English name).
# Markers carry a fixed model identity; the two numbers per row are the shortest and
# longest of the three model ranges.
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from utils.style import get_figure_size, save_figure, use_publication_style
from utils.terminology import BAND_LABELS, PALETTE_CATEGORICAL

use_publication_style()

# Fixed (marker, colour) per variogram model, and the range column it maps to.
MODEL_STYLE = {
    "spherical": ("o", "#B884A6"),
    "exponential": ("s", PALETTE_CATEGORICAL["blue"]),
    "gaussian": ("^", PALETTE_CATEGORICAL["orange"]),
}
RANGE_COL = {m: f"{m}.range_km" for m in MODELS}

# Rows top-to-bottom: the label; an illustrative sample of named predictors (terrain/access
# and optical/SAR, split by a dotted rule); and the XGBoost + TESSERA parcel residual.
LABEL_VARS = ["ogf_label"]
TERRAIN_ACCESS = [
    "elevation_m",
    "slope_deg",
    "dist_paved_road_m",
    "dist_unpaved_road_m",
    "dist_footpath_m",
]
OPTICAL_SAR = [
    "s2_blue",
    "s2_green",
    "s2_red",
    "s2_nir",
    "ndvi_p90",
    "ndvi_p50",
    "ndvi_p10",
    "s1_vv",
    "s1_vh",
    "s1_vh_vv_ratio",
]
PREDICTOR_VARS = TERRAIN_ACCESS + OPTICAL_SAR
RESIDUAL_VARS = ["xgboost__baseline_tessera__parcel"]
DISPLAY_NAME = {
    **BAND_LABELS,
    "ogf_label": "Old-growth labels",
    "xgboost__baseline_tessera__parcel": "XGBoost + TESSERA (parcel)",
}


def _draw_panel(ax, variables, *, sep_after=None):
    """Dumbbell rows top-to-bottom; sep_after draws a dotted rule below that many rows."""
    n = len(variables)
    for i, var in enumerate(variables):
        y = n - 1 - i  # first listed variable at the top
        vals = {m: float(summary.loc[var, RANGE_COL[m]]) for m in MODELS}
        lo, hi = min(vals.values()), max(vals.values())
        ax.plot([lo, hi], [y, y], color="0.5", lw=0.9, zorder=1)
        for m in MODELS:
            marker, colour = MODEL_STYLE[m]
            ax.scatter(vals[m], y, marker=marker, color=colour, s=24, zorder=3, linewidths=0)
        if hi - lo < 0.08:  # markers coincide: a single label is enough
            ax.annotate(
                f"{hi:.1f}",
                (hi, y),
                textcoords="offset points",
                xytext=(5, 0),
                ha="left",
                va="center",
                fontsize=6.5,
            )
        else:
            ax.annotate(
                f"{lo:.1f}",
                (lo, y),
                textcoords="offset points",
                xytext=(-5, 0),
                ha="right",
                va="center",
                fontsize=6.5,
            )
            ax.annotate(
                f"{hi:.1f}",
                (hi, y),
                textcoords="offset points",
                xytext=(5, 0),
                ha="left",
                va="center",
                fontsize=6.5,
            )
    ax.set_yticks(range(n))
    ax.set_yticklabels([DISPLAY_NAME.get(v, v) for v in reversed(variables)], fontsize=7)
    ax.set_ylim(-0.6, n - 0.4)
    if sep_after is not None:
        ax.axhline(n - sep_after - 0.5, color="0.7", lw=0.8, ls=(0, (1, 1)))
    ax.grid(axis="x", color="0.9", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=7)


plot_vars = LABEL_VARS + PREDICTOR_VARS + RESIDUAL_VARS
xmax = max(float(summary.loc[v, RANGE_COL[m]]) for v in plot_vars for m in MODELS)

w, h = get_figure_size("single", aspect=1.6)
fig, (ax_top, ax_mid, ax_bot) = plt.subplots(
    3,
    1,
    sharex=True,
    figsize=(1.6 * w, h),  # 80% of the earlier double width
    gridspec_kw={"height_ratios": [len(LABEL_VARS), len(PREDICTOR_VARS), len(RESIDUAL_VARS)]},
    constrained_layout=True,
)
_draw_panel(ax_top, LABEL_VARS)
_draw_panel(ax_mid, PREDICTOR_VARS, sep_after=len(TERRAIN_ACCESS))
_draw_panel(ax_bot, RESIDUAL_VARS)
ax_top.set_title("Labels", loc="center", fontweight="bold", fontsize=9)
ax_mid.set_title("Predictors (illustrative sample)", loc="center", fontweight="bold", fontsize=9)
ax_bot.set_title("Residuals", loc="center", fontweight="bold", fontsize=9)
ax_bot.set_xlim(0, xmax * 1.18)
ax_bot.set_xlabel("Effective variogram range (km)", fontsize=8)

handles = [
    Line2D([0], [0], marker=marker, color=colour, lw=0, label=model.capitalize())
    for model, (marker, colour) in MODEL_STYLE.items()
]
fig.legend(
    handles=handles,
    loc="outside lower center",
    ncol=3,
    frameon=False,
    fontsize=7.5,
    title="Variogram model",
    title_fontsize=7.5,
)
written = save_figure(
    fig, f"{NOTEBOOK}/range_summary", data=summary.loc[plot_vars, [RANGE_COL[m] for m in MODELS]]
)
plt.show()

## Spatial dependence summary

In [ ]:
# Spatial dependence summary. Single-column, two-panel figure (semivariogram range |
# Moran's I at 10 km) for the old-growth label and the named predictors, split into four
# stacked sub-plots: Labels, Baseline predictors, Conventional EO predictors and GFM predictors.
# A model whose range is unresolved (below the first lag class or into the max-lag
# boundary) is dropped; a row with none states the reason ("Unresolved: ...") on the semivariogram
# panel (Moran's I kept). Each foundation-model family collapses to one row: median (and
# IQR whisker) of the range / Moran's I across its embedding dimensions.
# Model residuals are shown separately in the next figure.
import textwrap

import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerBase
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.transforms import Bbox

from utils.style import get_figure_size, save_figure
from utils.terminology import PALETTE_CATEGORICAL

RANGE_KM = fits.pivot_table(index="variable", columns="model", values="range_km")
UNRESOLVED = fits.pivot_table(index="variable", columns="model", values="range_unresolved").astype(
    bool
)
MORAN10 = moran[moran["threshold_m"] == 10_000].set_index("variable")["morans_i"]

MODEL_STYLE = {
    "gaussian": ("^", PALETTE_CATEGORICAL["orange"]),
    "spherical": ("o", PALETTE_CATEGORICAL["magenta"]),
    "exponential": ("s", PALETTE_CATEGORICAL["blue"]),
}
PLOT_MODELS = list(MODEL_STYLE)
SUMMARY_COLOR = "black"

FS_TITLE, FS_SECTION, FS_YLBL, FS_VAL, FS_AXIS, FS_TICK, FS_LEG = 9.5, 8.5, 7.0, 6.4, 8.0, 7.0, 7.0
MARK_S, MORAN_MS, BOX_W = 11, 6, 0.5
RANGE_XMIN, RANGE_XMAX, RANGE_XLIM_R = -4.5, 25.0, 36.5  # room for the TESSERA whisker (35 km)
MORAN_XMIN, MORAN_XMAX = 0.0, 0.75

LABEL_ROWS = ["ogf_label"]
BASELINE = [
    "elevation_m",
    "slope_deg",
    "heat_load_index",
    "dist_paved_road_m",
    "dist_unpaved_road_m",
    "dist_footpath_m",
]
CONVENTIONAL_EO = [
    "s2_blue",
    "s2_green",
    "s2_red",
    "s2_nir",
    "swir_b11",
    "swir_b12",
    "ndvi_p90",
    "ndvi_p50",
    "ndvi_p10",
    "s1_vv",
    "s1_vh",
    "s1_vh_vv_ratio",
    "vpp_ampl",
    "vpp_eosd",
    "vpp_eosv",
    "vpp_lslope",
    "vpp_maxv",
    "vpp_minv",
    "vpp_rslope",
    "vpp_sosd",
    "vpp_sosv",
    "vpp_sprod",
]
GFM = [("summary", "alphaearth"), ("summary", "tessera")]
SECTIONS = [
    (LABEL_ROWS, "(a) Reference labels"),
    (BASELINE, "(b) Baseline predictors"),
    (CONVENTIONAL_EO, "(c) Conventional EO predictors"),
    (GFM, "(d) GFM embedding predictors"),
]

DISPLAY_NAME = {
    "ogf_label": "Old-growth/non-old-growth label",
    "elevation_m": "Elevation",
    "slope_deg": "Slope",
    "heat_load_index": "Heat load index",
    "dist_paved_road_m": "Dist. to paved road",
    "dist_unpaved_road_m": "Dist. to unpaved road",
    "dist_footpath_m": "Dist. to footpath",
    "s2_blue": "Sentinel-2 blue",
    "s2_green": "Sentinel-2 green",
    "s2_red": "Sentinel-2 red",
    "s2_nir": "Sentinel-2 NIR",
    "swir_b11": "SWIR band 11",
    "swir_b12": "SWIR band 12",
    "ndvi_p90": "NDVI p90",
    "ndvi_p50": "NDVI p50",
    "ndvi_p10": "NDVI p10",
    "s1_vv": "Sentinel-1 VV",
    "s1_vh": "Sentinel-1 VH",
    "s1_vh_vv_ratio": "Sentinel-1 VH/VV ratio",
    "vpp_ampl": "VPP amplitude",
    "vpp_eosd": "VPP EOS date",
    "vpp_eosv": "VPP EOS value",
    "vpp_lslope": "VPP left slope",
    "vpp_maxv": "VPP max value",
    "vpp_minv": "VPP min value",
    "vpp_rslope": "VPP right slope",
    "vpp_sosd": "VPP SOS date",
    "vpp_sosv": "VPP SOS value",
    "vpp_sprod": "VPP productivity",
}
SUMMARY_DISPLAY = {"alphaearth": "AlphaEarth", "tessera": "TESSERA v2"}


def _summary_quartiles(key):
    """Median + IQR of the resolved range (km) and Moran's I across a GFM family's dimensions."""
    variables = list(fits[(fits["group"] == key) & (~fits["is_pc"])]["variable"].unique())
    reps = [
        dd.loc[~dd["range_unresolved"], "range_km"].median()
        for v in variables
        for dd in [fits[fits["variable"] == v]]
        if (~dd["range_unresolved"]).any()
    ]
    range_vals = np.asarray(reps, dtype=float)
    moran_vals = MORAN10.reindex(variables).dropna().to_numpy()
    return {
        "range": np.percentile(range_vals, [25, 50, 75]),
        "moran": np.percentile(moran_vals, [25, 50, 75]),
        "range_values": range_vals,  # per-dimension values, for the box-plot variant
        "moran_values": moran_vals,
    }


SUMMARY = {key: _summary_quartiles(key) for _, key in GFM}


def _is_na(var):
    return isinstance(var, str) and bool(UNRESOLVED.loc[var].all())


def _unresolved_reason(var):
    """State why every model's range is unresolved for this row. The floor (a fit collapsing
    below the first lag bin) takes precedence: a pure-nugget variable pegs some models at the
    ceiling *and* collapses others to the floor, and the floor -- no spatial structure -- is a
    more faithful description than an implied long range."""
    maxlag_km = float(fits["maxlag_m"].max()) / 1000.0
    first_bin_km = maxlag_km / float(fits["n_lags"].max())
    ranges = [float(RANGE_KM.loc[var, m]) for m in PLOT_MODELS]
    if any(np.isfinite(r) and r <= first_bin_km for r in ranges):
        return "Unresolved (no spatial structure)"
    return f"Unresolved (no clear sill <{maxlag_km:.0f} km)"


def _wrap(text, width=20):
    return text if "\n" in text or len(text) <= width else "\n".join(textwrap.wrap(text, width))


def _finish_panel(ax, rows, xmin, xmax, is_moran):
    n = len(rows)
    ax.set_ylim(-0.5, n - 0.5)
    ax.set_xlim(xmin, xmax)
    ax.grid(axis="x", color="0.9", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=FS_TICK)
    if is_moran:  # no y tick marks on the Moran panel
        ax.tick_params(axis="y", which="both", left=False, labelleft=False)


def _box(ax, values, y):
    """Tukey box for a GFM family at row `y`: interquartile box, median line and 1.5 x IQR
    whiskers, one row high, so it occupies the space a single variable row would. Fliers are
    not drawn -- the panel's x-limit is set by the variable rows, so a far-out dimension would
    be clipped at the spine rather than shown."""
    ax.boxplot(
        [np.asarray(values, dtype=float)],
        positions=[y],
        widths=BOX_W,
        orientation="horizontal",
        showfliers=False,
        manage_ticks=False,
        patch_artist=True,
        boxprops={"facecolor": "white", "edgecolor": SUMMARY_COLOR, "linewidth": 0.7},
        medianprops={"color": SUMMARY_COLOR, "linewidth": 1.0},
        whiskerprops={"color": SUMMARY_COLOR, "linewidth": 0.7},
        capprops={"color": SUMMARY_COLOR, "linewidth": 0.7},
        zorder=3,
    )


class _BoxPlotKey(HandlerBase):
    """Legend key drawn as a miniature box plot -- capped whiskers, IQR box and median line --
    so the key matches the glyph used for the collapsed GFM rows."""

    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, t):
        x0, yc = -xdescent, height / 2 - ydescent
        box_l, box_r, med = x0 + 0.26 * width, x0 + 0.70 * width, x0 + 0.44 * width
        half_box, half_cap = 0.34 * height, 0.24 * height
        lw = 0.7
        parts = [
            Line2D([x0, box_l], [yc, yc], color=SUMMARY_COLOR, lw=lw),
            Line2D([box_r, x0 + width], [yc, yc], color=SUMMARY_COLOR, lw=lw),
            Line2D([x0, x0], [yc - half_cap, yc + half_cap], color=SUMMARY_COLOR, lw=lw),
            Line2D(
                [x0 + width, x0 + width],
                [yc - half_cap, yc + half_cap],
                color=SUMMARY_COLOR,
                lw=lw,
            ),
            Rectangle(
                (box_l, yc - half_box),
                box_r - box_l,
                2 * half_box,
                facecolor="white",
                edgecolor=SUMMARY_COLOR,
                linewidth=lw,
            ),
            Line2D([med, med], [yc - half_box, yc + half_box], color=SUMMARY_COLOR, lw=1.0),
        ]
        for part in parts:
            part.set_transform(t)
        return parts


def _draw_range(ax, rows):
    n = len(rows)
    for i, var in enumerate(rows):
        y = n - 1 - i
        if isinstance(var, tuple):  # collapsed GFM family
            _box(ax, SUMMARY[var[1]]["range_values"], y)
            continue
        res = [float(RANGE_KM.loc[var, m]) for m in PLOT_MODELS if not bool(UNRESOLVED.loc[var, m])]
        if not res:  # range not resolved by any model
            ax.text(
                RANGE_XMIN + 0.2,
                y,
                _unresolved_reason(var),
                ha="left",
                va="center_baseline",
                fontsize=FS_VAL,
                color="black",
            )
            continue
        lo, hi = min(res), max(res)
        ax.plot([lo, hi], [y, y], color="0.45", lw=0.9, zorder=1)
        for m in PLOT_MODELS:
            if bool(UNRESOLVED.loc[var, m]):
                continue
            marker, colour = MODEL_STYLE[m]
            ax.scatter(
                float(RANGE_KM.loc[var, m]),
                y,
                marker=marker,
                color=colour,
                s=MARK_S,
                zorder=3,
                edgecolors="0.25",
                linewidths=0.35,
            )
        if f"{lo:.1f}" == f"{hi:.1f}":  # only one distinct value available
            ax.annotate(
                f"{hi:.1f}",
                (hi, y),
                textcoords="offset points",
                xytext=(5, 0),
                ha="left",
                va="center_baseline",
                fontsize=FS_VAL,
                clip_on=False,
            )
        else:  # always show both the min (left) and the max (right)
            ax.annotate(
                f"{lo:.1f}",
                (lo, y),
                textcoords="offset points",
                xytext=(-5, 0),
                ha="right",
                va="center_baseline",
                fontsize=FS_VAL,
                clip_on=False,
            )
            ax.annotate(
                f"{hi:.1f}",
                (hi, y),
                textcoords="offset points",
                xytext=(5, 0),
                ha="left",
                va="center_baseline",
                fontsize=FS_VAL,
                clip_on=False,
            )
    _finish_panel(ax, rows, RANGE_XMIN, RANGE_XLIM_R, is_moran=False)


def _draw_moran(ax, rows):
    n = len(rows)
    for i, var in enumerate(rows):
        y = n - 1 - i
        if isinstance(var, tuple):
            _box(ax, SUMMARY[var[1]]["moran_values"], y)
            continue
        v = float(MORAN10.loc[var])
        ax.plot([0, v], [y, y], color="black", lw=0.5, zorder=1)
        ax.scatter(v, y, marker="D", color="black", s=MORAN_MS, zorder=3)
        ax.annotate(
            f"{v:.2f}",
            (v, y),
            textcoords="offset points",
            xytext=(4, 0),  # always right
            ha="left",
            va="center_baseline",
            fontsize=FS_VAL,
            clip_on=False,
        )
    _finish_panel(ax, rows, MORAN_XMIN, MORAN_XMAX, is_moran=True)


def _set_yticks(ax, rows):
    n = len(rows)
    pos, lab = [], []
    for i, var in enumerate(rows):
        pos.append(n - 1 - i)
        lab.append(SUMMARY_DISPLAY[var[1]] if isinstance(var, tuple) else _wrap(DISPLAY_NAME[var]))
    ax.set_yticks(pos)
    ticklabels = ax.set_yticklabels(lab, fontsize=FS_YLBL)
    for tick, text in zip(ticklabels, lab, strict=False):  # single-line optical / multi-line block
        tick.set_va("center" if "\n" in text else "center_baseline")


fig_w, _ = get_figure_size("single")  # single journal column (90 mm)
fig = plt.figure(figsize=(fig_w, 7.4))
gs = fig.add_gridspec(
    len(SECTIONS),
    2,
    height_ratios=[len(rows) for rows, _ in SECTIONS],
    width_ratios=[2.0, 1.2],
    left=0.30,
    right=0.968,
    top=0.978,
    bottom=0.135,
    hspace=0.26,
    wspace=0.06,
)
axes = []
for r, (rows, _title) in enumerate(SECTIONS):
    ax_r = fig.add_subplot(gs[r, 0])
    ax_m = fig.add_subplot(gs[r, 1])
    _draw_range(ax_r, rows)
    _draw_moran(ax_m, rows)
    _set_yticks(ax_r, rows)
    ax_m.set_yticks([])
    axes.append((ax_r, ax_m))

for ax_r, ax_m in axes:  # identical tick positions on every subplot so the gridlines line up
    ax_r.set_xticks([0, 5, 10, 15, 20, 25, 30, 35])
    ax_m.set_xticks([0.0, 0.2, 0.4, 0.6])
for ax_r, ax_m in axes[:-1]:  # only the bottom section shows the tick labels + axis titles
    ax_r.set_xticklabels([])
    ax_m.set_xticklabels([])
axes[-1][0].set_xticklabels(
    ["0", "5", "10", "15", "20", "25", "30", ""]
)  # 35: tick and gridline only
axes[-1][0].set_xlabel("Semivariogram range (km)", fontsize=FS_AXIS)
axes[-1][1].set_xlabel("Moran\u2019s I at 10 km", fontsize=FS_AXIS)

fig.canvas.draw()  # section titles left-aligned to the semivariogram panel's left edge
for (ax_r, _ax_m), (_rows, title) in zip(axes, SECTIONS, strict=False):
    fig.text(
        ax_r.get_position().x0,
        ax_r.get_position().y1 + 0.004,
        title,
        ha="left",
        va="bottom",
        fontsize=FS_SECTION,
    )

# The GFM-summary handle is a box patch, matching the collapsed AlphaEarth / TESSERA rows.
h_box = Patch(
    facecolor="white",
    edgecolor=SUMMARY_COLOR,
    linewidth=0.7,
    label="Box plot over all\nresolved embeddings",
)
# Column-major handle order so row 1 = the three ranges, row 2 = Moran diamond / median summary.
handles = [
    Line2D(
        [0],
        [0],
        marker="^",
        color=PALETTE_CATEGORICAL["orange"],
        lw=0,
        markeredgecolor="0.25",
        markeredgewidth=0.4,
        label="Range (Gaussian)",
        markersize=5.5,
    ),
    Line2D(
        [0], [0], marker="D", color="black", lw=0, label="Moran\u2019s I at 10 km", markersize=3.5
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color=PALETTE_CATEGORICAL["magenta"],
        lw=0,
        markeredgecolor="0.25",
        markeredgewidth=0.4,
        label="Range (spherical)",
        markersize=5.5,
    ),
    h_box,
    Line2D(
        [0],
        [0],
        marker="s",
        color=PALETTE_CATEGORICAL["blue"],
        lw=0,
        markeredgecolor="0.25",
        markeredgewidth=0.4,
        label="Range (exponential)",
        markersize=5.5,
    ),
]
# Lifted a little above y = 0 with no border padding; the crop below follows the legend.
fig.legend(
    handles=handles,
    handler_map={h_box: _BoxPlotKey()},
    loc="lower center",
    bbox_to_anchor=(0.5, 0.012),
    ncol=3,
    frameon=False,
    fontsize=FS_LEG,
    handletextpad=0.4,
    columnspacing=0.7,
    borderpad=0.0,
    labelspacing=0.85,  # the bottom row stays anchored; this lifts the top row off it
)

# Data sidecar: per-model resolved ranges (NaN where unresolved) for variable rows; range and
# Moran quartiles for the collapsed GFM-family summary rows.
rows_out = []
for var in LABEL_ROWS + BASELINE + CONVENTIONAL_EO + GFM:
    if isinstance(var, tuple):
        rq, mq = SUMMARY[var[1]]["range"], SUMMARY[var[1]]["moran"]
        rows_out.append(
            {
                "variable": var[1],
                "kind": "summary_median_iqr",
                "range_q1_km": rq[0],
                "range_median_km": rq[1],
                "range_q3_km": rq[2],
                "moran_q1": mq[0],
                "moran_median": mq[1],
                "moran_q3": mq[2],
            }
        )
    else:
        entry = {
            "variable": var,
            "kind": "range_not_resolved" if _is_na(var) else "variable",
            "moran_I_10km": float(MORAN10.loc[var]),
        }
        for m in PLOT_MODELS:
            entry[f"{m}_range_km"] = (
                np.nan if bool(UNRESOLVED.loc[var, m]) else float(RANGE_KM.loc[var, m])
            )
        rows_out.append(entry)
fig_data = pd.DataFrame(rows_out)

# Crop to the full single-column width (keeps the legend centred at x=0.5) but trim
# vertically: a hair of air below the legend (its two-line label's descenders otherwise
# touch the page edge) and above the first sub-plot.
fig.canvas.draw()
_tight = fig.get_tightbbox(fig.canvas.get_renderer())
_crop = Bbox.from_extents(0.0, _tight.y0 - 0.03, fig.get_size_inches()[0], _tight.y1 + 0.02)

written = save_figure(fig, f"{NOTEBOOK}/fig_2_spatial_dependence_summary", data=fig_data)
fig.savefig(written[0], bbox_inches=_crop)
print(
    f"[figure] {written[0].relative_to(paths.repo_root)}; "
    f"AlphaEarth {SUMMARY['alphaearth']['range'][1]:.1f} km / "
    f"TESSERA {SUMMARY['tessera']['range'][1]:.1f} km"
)
plt.show()

## Residual spatial dependence by feature group

In [ ]:
# Residual spatial dependence, faceted by feature group. One column per feature set; top row =
# semivariogram range (three-model dumbbell, annotated with min/max), bottom row = Moran's I at
# 10 km (annotated with its value); rows within a subplot are the four architectures. Each of
# the eight subplots has its own (adjustable) x-axis. Parcel-level out-of-fold residuals only.
RES_RANGE = fits.pivot_table(index="variable", columns="model", values="range_km")
RES_UNRES = fits.pivot_table(index="variable", columns="model", values="range_unresolved").astype(
    bool
)
RES_M10 = moran[moran["threshold_m"] == 10_000].set_index("variable")["morans_i"]

R_MODEL_STYLE = {
    "gaussian": ("^", PALETTE_CATEGORICAL["orange"]),
    "spherical": ("o", "#B884A6"),
    "exponential": ("s", PALETTE_CATEGORICAL["blue"]),
}
R_MODELS = list(R_MODEL_STYLE)
RF_TITLE, RF_COL, RF_YLBL, RF_AXIS, RF_TICK, RF_VAL, RF_LEG = 11, 8.5, 8.0, 9.0, 7.5, 6.5, 8.0
R_MARK_S, R_MORAN_MS = 22, 16
R_RANGE_XLIM, R_RANGE_TICKS = (2.0, 10.0), [2, 4, 6, 8, 10]  # shared range scale (all columns)
R_MORAN_PAD = 1.4  # right-hand headroom so the Moran value labels fit

FEATURE_GROUPS = [
    ("baseline", "Baseline"),
    ("baseline_conventional_eo", "Baseline + Conventional EO"),
    ("baseline_alphaearth", "Baseline + AlphaEarth"),
    ("baseline_tessera", "Baseline + TESSERA"),
]
ARCHES = [
    ("xgboost", "XGBoost"),
    ("cnn_3x3", "CNN 3x3"),
    ("cnn_5x5", "CNN 5x5"),
    ("cnn_7x7", "CNN 7x7"),
]
n_arch = len(ARCHES)

# One Moran scale for every column, from the global maximum across all model/feature combinations.
R_MORAN_XLIM = (
    0.0,
    max(
        float(RES_M10.loc[f"{arch}__{fg_key}__parcel"])
        for fg_key, _ in FEATURE_GROUPS
        for arch, _ in ARCHES
    )
    * R_MORAN_PAD,
)

double_w, _ = get_figure_size("double")  # multi-panel results figure: full page width
fig_r, r_axes = plt.subplots(2, len(FEATURE_GROUPS), sharey=True, figsize=(double_w, 3.70))

for j, (fg_key, fg_label) in enumerate(FEATURE_GROUPS):
    ax_range, ax_moran = r_axes[0, j], r_axes[1, j]
    per_arch = []
    for i, (arch, _arch_label) in enumerate(ARCHES):
        var = f"{arch}__{fg_key}__parcel"
        res = [
            (m, float(RES_RANGE.loc[var, m])) for m in R_MODELS if not bool(RES_UNRES.loc[var, m])
        ]
        per_arch.append((i, res, float(RES_M10.loc[var])))
    # both metrics on a shared scale across all four columns
    ax_range.set_xlim(*R_RANGE_XLIM)
    ax_moran.set_xlim(*R_MORAN_XLIM)
    for i, res, mv in per_arch:
        y = n_arch - 1 - i
        if res:
            lo, hi = min(v for _, v in res), max(v for _, v in res)
            ax_range.plot([lo, hi], [y, y], color="0.45", lw=0.9, zorder=1)
            for m, v in res:
                marker, colour = R_MODEL_STYLE[m]
                ax_range.scatter(
                    v,
                    y,
                    marker=marker,
                    color=colour,
                    s=R_MARK_S,
                    zorder=3,
                    edgecolors="0.25",
                    linewidths=0.35,
                )
            if f"{lo:.1f}" == f"{hi:.1f}":  # only one distinct value
                ax_range.annotate(
                    f"{hi:.1f}",
                    (hi, y),
                    textcoords="offset points",
                    xytext=(4, 0),
                    ha="left",
                    va="center",
                    fontsize=RF_VAL,
                )
            else:  # min (left) and max (right)
                ax_range.annotate(
                    f"{lo:.1f}",
                    (lo, y),
                    textcoords="offset points",
                    xytext=(-4, 0),
                    ha="right",
                    va="center",
                    fontsize=RF_VAL,
                )
                ax_range.annotate(
                    f"{hi:.1f}",
                    (hi, y),
                    textcoords="offset points",
                    xytext=(4, 0),
                    ha="left",
                    va="center",
                    fontsize=RF_VAL,
                )
        ax_moran.plot([0, mv], [y, y], color="0.6", lw=0.8, zorder=1)
        ax_moran.scatter(mv, y, marker="D", color="black", s=R_MORAN_MS, zorder=3)
        ax_moran.annotate(
            f"{mv:.2f}",
            (mv, y),
            textcoords="offset points",
            xytext=(4, 0),
            ha="left",
            va="center",
            fontsize=RF_VAL,
        )
    ax_range.set_title(fg_label, fontsize=RF_COL, loc="left")
    ax_range.set_xticks(R_RANGE_TICKS)
    ax_moran.locator_params(axis="x", nbins=4)
    for ax in (ax_range, ax_moran):
        ax.set_ylim(-0.6, n_arch - 0.4)
        ax.grid(axis="x", color="0.9", lw=0.6)
        ax.set_axisbelow(True)
        ax.tick_params(labelsize=RF_TICK)

r_axes[0, 0].set_yticks(range(n_arch))
r_axes[0, 0].set_yticklabels([label for _, label in reversed(ARCHES)], fontsize=RF_YLBL)

fig_r.subplots_adjust(left=0.10, right=0.985, top=0.90, bottom=0.235, hspace=0.6, wspace=0.16)

# One centred x-axis label per metric row.
fig_r.canvas.draw()
_p0, _p3 = r_axes[0, 0].get_position(), r_axes[0, 3].get_position()
_cx = (_p0.x0 + _p3.x1) / 2
fig_r.text(
    _cx,
    r_axes[0, 0].get_position().y0 - 0.06,
    "Semivariogram range (km)",
    ha="center",
    va="top",
    fontsize=RF_AXIS,
)
fig_r.text(
    _cx,
    r_axes[1, 0].get_position().y0 - 0.085,
    "Moran's I at 10 km",
    ha="center",
    va="top",
    fontsize=RF_AXIS,
)

r_handles = [
    Line2D(
        [0],
        [0],
        marker="^",
        color=PALETTE_CATEGORICAL["orange"],
        lw=0,
        markeredgecolor="0.25",
        markeredgewidth=0.4,
        label="Range (Gaussian)",
        markersize=6,
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="#B884A6",
        lw=0,
        markeredgecolor="0.25",
        markeredgewidth=0.4,
        label="Range (spherical)",
        markersize=6,
    ),
    Line2D(
        [0],
        [0],
        marker="s",
        color=PALETTE_CATEGORICAL["blue"],
        lw=0,
        markeredgecolor="0.25",
        markeredgewidth=0.4,
        label="Range (exponential)",
        markersize=6,
    ),
    Line2D([0], [0], marker="D", color="black", lw=0, label="Moran's I at 10 km", markersize=5.5),
]
fig_r.legend(
    handles=r_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=4,
    frameon=False,
    fontsize=RF_LEG,
    columnspacing=1.6,
)

written_r = save_figure(fig_r, f"{NOTEBOOK}/residual_spatial_dependence")
print(
    f"[figure] {written_r[0].relative_to(paths.repo_root)}; "
    f"{n_arch * len(FEATURE_GROUPS)} model combinations"
)
plt.show()

## Diagnostic: empirical and fitted semivariograms

A small-multiples panel of the **empirical** variogram (grey points) and the three **fitted**
theoretical models (Gaussian, spherical, exponential curves) for every label and
non-foundation predictor — the baseline terrain/access bands and the conventional-EO
optical/SAR/phenology bands, excluding the AlphaEarth and TESSERA embeddings (whose
dimensions have no plain-English name).

In [ ]:
# Semivariogram diagnostics. Empirical points + three fitted model curves per variable,
# for the label and all non-foundation predictors. Fits with an unresolved range are dotted.
empirical = pd.read_csv(EMPIRICAL_CSV)
fitted = pd.read_csv(FITTED_CSV)

NONFOUNDATION_GROUPS = ["label", "baseline", "conventional_eo"]
diag_vars = list(
    fits[fits["group"].isin(NONFOUNDATION_GROUPS)].drop_duplicates("variable")["variable"]
)
unres_by_var = {
    var: sub.set_index("model")["range_unresolved"].to_dict()
    for var, sub in fits[fits["variable"].isin(diag_vars)].groupby("variable")
}
DIAG_STYLE = {
    "gaussian": PALETTE_CATEGORICAL["orange"],
    "spherical": "#B884A6",
    "exponential": PALETTE_CATEGORICAL["blue"],
}
DIAG_DISPLAY = {**BAND_LABELS, "ogf_label": "Old-growth labels"}
DIAG_MAXLAG_KM = float(fits["maxlag_m"].max()) / 1000

n_col = 5
n_row = int(np.ceil(len(diag_vars) / n_col))
diag_w, _ = get_figure_size("double")  # multi-panel diagnostic: full page width
fig_d, axgrid = plt.subplots(n_row, n_col, figsize=(diag_w, 1.55 * n_row), squeeze=False)

for idx, var in enumerate(diag_vars):
    ax = axgrid[idx // n_col][idx % n_col]
    emp_v = empirical[empirical["variable"] == var].sort_values("lag_m")
    ax.scatter(emp_v["lag_m"] / 1000, emp_v["semivariance"], s=4, color="0.3", zorder=3)
    vunres = unres_by_var.get(var, {})
    for model, colour in DIAG_STYLE.items():
        curve = fitted[(fitted["variable"] == var) & (fitted["model"] == model)].sort_values(
            "lag_m"
        )
        style = ":" if vunres.get(model, False) else "-"
        ax.plot(
            curve["lag_m"] / 1000, curve["semivariance"], color=colour, lw=1.1, ls=style, zorder=2
        )
    resolved_here = any(not u for u in vunres.values())
    ax.set_title(
        DIAG_DISPLAY.get(var, var) + ("" if resolved_here else "  (range not resolved)"),
        fontsize=7,
        fontweight="bold" if resolved_here else "normal",
    )
    ax.set_xlim(0, DIAG_MAXLAG_KM)
    ax.set_ylim(bottom=0)
    ax.tick_params(labelsize=6)
    ax.yaxis.get_offset_text().set_fontsize(5.5)
    ax.grid(color="0.92", lw=0.5)
    ax.set_axisbelow(True)

for j in range(len(diag_vars), n_row * n_col):  # hide unused cells
    axgrid[j // n_col][j % n_col].axis("off")

fig_d.supxlabel("Lag distance (km)", fontsize=9)
fig_d.supylabel("Semivariance", fontsize=9)
diag_handles = [Line2D([0], [0], marker="o", color="0.3", lw=0, markersize=4, label="Empirical")]
diag_handles += [
    Line2D([0], [0], color=c, lw=1.4, label=m.capitalize()) for m, c in DIAG_STYLE.items()
]
diag_handles.append(
    Line2D([0], [0], color="0.4", lw=1.1, ls=":", label="Fit with range not resolved")
)
fig_d.legend(
    handles=diag_handles,
    loc="upper center",
    ncol=5,
    frameon=False,
    fontsize=8,
    bbox_to_anchor=(0.5, 1.0),
)
fig_d.suptitle(
    "Empirical and fitted semivariograms: labels and non-foundation predictors", fontsize=11, y=1.02
)
fig_d.tight_layout(rect=(0.03, 0.02, 1, 0.985))

written_d = save_figure(fig_d, f"{NOTEBOOK}/semivariogram_diagnostics")
print(f"[figure] {written_d[0].relative_to(paths.repo_root)}; {len(diag_vars)} variables")
plt.show()